# 4. HLS: C++ into a circuit (the naive version)

**High-Level Synthesis** compiles C++ into RTL. You write the convolution roughly the
way a software person would - loop over output pixels, loop over the 3x3 window,
accumulate - add a few `#pragma`s, and the tool builds the datapath, the pipeline, and
an **AXI master** so the accelerator reads the image straight out of DRAM itself (no more
per-pixel MMIO from the CPU). That is `hls/conv3x3.cpp`.

### Initiation Interval (II)

The key number for a pipelined hardware loop is its **II**: how many clock cycles between
successive iterations starting. II=1 means a new output pixel every clock - the goal.
The naive loop re-reads its 3x3 window from DRAM for *every* output pixel - up to nine
reads - and those reads all go through one memory port. The tool cannot start a new
iteration until the previous one's reads finish, so it reports **II=9**: nine clocks per
pixel. The datapath could go nine times faster; memory access is holding it back.

> **Board only.** The rungs below need the PYNQ board (`pynq` + the bitstream). On a
> laptop `available_backends()` omits them and the cell prints a skip message - that is
> expected; run this one on the board.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from fpga_conv import KERNELS, get_kernel, conv_reference, to_grayscale_u8, available_backends
from fpga_conv.bench import (
    Scoreboard, run_rung, time_backend, sample_gray, make_sample_image, roofline,
)

# Path to the combined bitstream (carries all four accelerators). Its .hwh must sit
# beside it. Relative to this notebook on the dev box; on the board point it at wherever
# you copied the build, e.g. '/home/xilinx/workshop/conv3x3.bit'.
BIT_PATH = os.path.abspath('../../rtl/build/conv3x3.bit')
HW_KW = {'bitfile': BIT_PATH}          # passed to the hardware rungs only

# The scoreboard persists across the whole series (notebooks/scoreboard.json), so the
# roofline grows as you climb. The hardware rungs are skipped off the board.
sb = Scoreboard()
print('backends available here:', available_backends())

In [ ]:
if 'hls_naive' in available_backends():
    img = sample_gray(256)
    out, t = run_rung('hls_naive', img, 'edges', repeats=10, scoreboard=sb, **HW_KW)
else:
    print('hls_naive backend not available here (laptop) - run this cell on the board.')

Faster than the MMIO RTL path - the accelerator now fetches its own data in bulk instead
of waiting on the CPU. But it is still far from the ceiling, because of those redundant
reads: 9 bytes read + 1 written per output pixel. That is a *low arithmetic intensity*
(few MACs per byte moved), which on the roofline puts this dot well to the **left**,
pinned under the memory-bandwidth slope rather than the compute ceiling.

In [ ]:
sb.roofline(); plt.show()

**The fix is the same idea the hand-written RTL already used:** don't re-read. Keep the
last two image rows on-chip so each pixel is read from DRAM exactly once.
-> `05_line_buffer.ipynb`.